# **EfficientNet Pre-trained Binary Model**

In this notebook, we train the final EfficientNet Pre-trained Binary Model and make predictions on the test set.

## Libraries

In [1]:
import os
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

In [2]:
# Libraries
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, precision_score, recall_score

from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.metrics import AUC
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from deep.constants import ROOT, METADATA_FILE, SEEDS, METADATA_DIR, INPUT_DIR, MODEL_IMAGE_SIZE, BATCH_SIZE
from deep.modelling.model_specifications import efficient_net_extended
from deep.modelling.pipiline_utils import split_data
from deep.preprocess.album_augmenter import compute_effective_class_weights
from deep.modelling.metric_utils import get_fitted_model_metrics, plot_confusion_matrix, plot_metrics

2025-05-01 00:39:43.492719: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-01 00:39:43.505204: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-01 00:39:43.521206: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-01 00:39:43.525542: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-01 00:39:43.536841: I tensorflow/core/platform/cpu_feature_guar

In [3]:
# Setting options
pd.set_option('display.max_rows', None)

## Data Loading

In [4]:
# Load the metadata
data = pd.read_csv(METADATA_FILE)

# Drop unnecessary columns for this problem
data.drop(columns=['phylum', 'family'], inplace=True)

## Data Preprocessing

In [5]:
# Define seeds and path
binary_baseline_path = f"{ROOT}/models/binary_scores.json"

### Clean Images

In [ ]:
# Create transformed images
notebook_path = f'"{ROOT}/notebooks/preprocess_routine/cleaner_routine.ipynb"'
%run $notebook_path

### Upsample Images

In [ ]:
# # DISCLAIMER: Ensure you are using 'json_path = UPSAMPLE_JSONS / "oversample_plan_20250427T092327Z.json"'

# # Create upsamples images
# notebook_path = f'"{ROOT}/notebooks/preprocess_routine/augmenter_routine.ipynb"'
# %run $notebook_path

### Resize Images

In [6]:
# Resize images
notebook_path = f'"{ROOT}/notebooks/preprocess_routine/resize_routine.ipynb"'
%run $notebook_path

I0000 00:00:1745884521.020544   22545 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1745884521.091320   22545 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1745884521.091512   22545 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1745884521.092406   22545 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

## Data Modelling

In [ ]:
# Defining the training and test sets
train_df, test_df = split_data(data, 'is_animal', SEEDS[0], val_size=0, save_split=False)
    
# # Get the upsampled images - cropped and generated
# cropped = pd.read_csv(f'{METADATA_DIR}/cropped_labels.csv')
# upsampled = pd.read_csv(f'{METADATA_DIR}/is_animal_upsample.csv')

# # Filter upsample based on the images on the train set
# # Upsampled images from crops
# aux_df1 = upsampled[upsampled['file_path'].str.contains('noanimalcrop', na=False)]

# # Upsampled images from train
# # Exclude the cropped images first
# temp = upsampled[~upsampled['file_path'].isin(aux_df1['file_path'])]

# # Filter to only rare_species_id in train_df
# aux_df2 = temp[temp['rare_species_id'].isin(train_df['rare_species_id'])]

# # Create new upsampled dataframe
# upsampled = pd.concat([
#     aux_df1
#     ,aux_df2
# ], ignore_index=True
# , axis=0)

# # Add the new metadata to train_df
# train_df = pd.concat([
#     train_df
#     ,cropped
#     ,upsampled
# ], ignore_index=True
# , axis=0)

In [7]:
# Changing target to string
train_df['is_animal'] = train_df['is_animal'].astype(str)
test_df['is_animal'] = test_df['is_animal'].astype(str)

In [8]:
train_datagen = ImageDataGenerator(
    rotation_range=90,
    shear_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    channel_shift_range=30.0,
    zoom_range=(0.8, 1.2),
    fill_mode='nearest',
    rescale=1./255
)
test_datagen = ImageDataGenerator(rescale=1./255)

# Train generator
binary_train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=INPUT_DIR,
    x_col='file_path',
    y_col='is_animal',
    target_size=MODEL_IMAGE_SIZE['efficientnetb4'],
    batch_size=BATCH_SIZE,
    class_mode='binary',
    seed=SEEDS[0],
    shuffle=True
)

# Test generator
binary_test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=INPUT_DIR,
    x_col='file_path',
    y_col='is_animal',
    target_size=MODEL_IMAGE_SIZE['efficientnetb4'],
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 17121 validated image filenames belonging to 2 classes.
Found 2397 validated image filenames belonging to 2 classes.


/home/shadybea/anaconda3/envs/dl/lib/python3.11/site-packages/keras/src/legacy/preprocessing/image.py:920: UserWarning: Found 2 invalid image filename(s) in x_col="file_path". These filename(s) will be ignored.
  warnings.warn(


In [9]:
# Compute class weights
class_weights = compute_effective_class_weights(train_df, 'is_animal')

In [11]:
# Defining the model
binary_model, _ = efficient_net_extended(regularizer=False, dropout=False, task_type='binary')

# Compiling the model
binary_model.compile(
    optimizer=RMSprop(learning_rate=0.001)
    ,loss='binary_crossentropy'
    ,metrics=['accuracy', AUC(), 'precision', 'recall']
)

# Fit the model
fitted_binary = binary_model.fit(
    binary_train_generator
    ,epochs=20
    ,steps_per_epoch=int(np.ceil(len(train_df) / BATCH_SIZE))
    ,class_weight=class_weights
    ,callbacks=[
        ReduceLROnPlateau(patience=2, factor=0.5, verbose=1)
    ]
    ,verbose=1
)

/home/shadybea/anaconda3/envs/dl/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20


I0000 00:00:1746056445.908831   21610 service.cc:146] XLA service 0x737510003d50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1746056445.908875   21610 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce MX350, Compute Capability 6.1
2025-05-01 00:40:46.554063: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-05-01 00:40:48.972282: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90100
2025-05-01 00:40:54.622645: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:310] gpu_async_0 cuMemAllocAsync failed to allocate 1437078016 bytes: CUDA error: out of memory (CUDA_ERROR_OUT_OF_MEMORY)
 Reported by CUDA: Free memory/Total memory: 249167872/2091253760
2025-05-01 00:40:54.622685: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc

536/536 ━━━━━━━━━━━━━━━━━━━━ 712s 1s/step - accuracy: 0.5867 - auc: 0.6151 - loss: 0.6739 - precision: 0.5901 - recall: 0.6474 - learning_rate: 0.0010
Epoch 2/20


/home/shadybea/anaconda3/envs/dl/lib/python3.11/site-packages/keras/src/callbacks/callback_list.py:145: UserWarning: Learning rate reduction is conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,auc,loss,precision,recall,learning_rate.
  callback.on_epoch_end(epoch, logs)


536/536 ━━━━━━━━━━━━━━━━━━━━ 685s 1s/step - accuracy: 0.6506 - auc: 0.6983 - loss: 0.6294 - precision: 0.6797 - recall: 0.6365 - learning_rate: 0.0010
Epoch 3/20
536/536 ━━━━━━━━━━━━━━━━━━━━ 681s 1s/step - accuracy: 0.6711 - auc: 0.7197 - loss: 0.6130 - precision: 0.7069 - recall: 0.6358 - learning_rate: 0.0010
Epoch 4/20
536/536 ━━━━━━━━━━━━━━━━━━━━ 685s 1s/step - accuracy: 0.6748 - auc: 0.7272 - loss: 0.6087 - precision: 0.7072 - recall: 0.6497 - learning_rate: 0.0010
Epoch 5/20
536/536 ━━━━━━━━━━━━━━━━━━━━ 681s 1s/step - accuracy: 0.6826 - auc: 0.7335 - loss: 0.6037 - precision: 0.7092 - recall: 0.6550 - learning_rate: 0.0010
Epoch 6/20
536/536 ━━━━━━━━━━━━━━━━━━━━ 679s 1s/step - accuracy: 0.6800 - auc: 0.7322 - loss: 0.6049 - precision: 0.7106 - recall: 0.6501 - learning_rate: 0.0010
Epoch 7/20
536/536 ━━━━━━━━━━━━━━━━━━━━ 678s 1s/step - accuracy: 0.6858 - auc: 0.7378 - loss: 0.6004 - precision: 0.7190 - recall: 0.6626 - learning_rate: 0.0010
Epoch 8/20
536/536 ━━━━━━━━━━━━━━━━━━━━

In [12]:
get_fitted_model_metrics(model=fitted_binary, is_final=True)

Train Loss: 0.5855104327201843
Train Precision:0.723846435546875


In [ ]:
# Make predictions
predictions = binary_model.predict(
    binary_test_generator
    ,steps=len(binary_test_generator)
    ,verbose=1
)

In [ ]:
# Get the true labels
y_true = binary_test_generator.labels

# Convert predictions to class labels
y_pred = (predictions > 0.5).astype(int).flatten()

# Calculate metrics
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall: {recall_score(y_true, y_pred):.4f}")

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(y_true, y_pred, data['is_animal'].unique())